# ST Guitar — Stage 7G-E3-S0-B Event-Level Error Attribution Audit

Amaç: S0'daki 399 final-epoch OOF kararını event düzeyinde yeniden üretmek ve özellikle FP/FN olaylarını **descriptive, target-blind geometry** eksenlerinde incelemek.

**Bu bir yeni model deneyi değildir.** Scheduler/tuning/yeni feature/yeni threshold/specialist training/architecture activation/early stopping/checkpoint/E3-E/Stage7E/production yasaktır.

Attribution etiketleri **nedensel Teacher-GOLD açıklaması değildir** ve specialist supervision üretmez.


In [ ]:
# STEP 1 — EXACT DEPENDENCIES + CLEAN RUNTIME RESTART
import sys, subprocess
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
    "numpy==2.4.6", "scipy==1.17.1", "scikit-learn==1.9.0",
    "joblib==1.5.3", "threadpoolctl==3.6.0"
], check=True)
print("===== DEPENDENCIES INSTALLED =====")
print("Runtime şimdi otomatik yeniden başlayacak. Sonra STEP 2 hücresinden devam et.")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
# STEP 2 — CANONICAL S0-B SETUP (restart sonrası)
PINNED_CODE_SHA="e8925f4e70b2099012c4e0ad164e5cca139f1ce6"
ANIMETAB_COMMIT="18c0993cbe0a0948cbf0b7768bcb09ff81c23a9a"
EXPECTED_CHOICES_SHA256="db0e752ec7b9e0e1b333a217d904175f4e57cd89a32b2511330ebab7b8c6c12e"
EXPECTED_S0_RESULT_SHA256="59238bb0c570ce9eb0294561814508397a23e15aa2cf47f1e9d9f9c4d4bded92"
EXPECTED_NUMPY="2.4.6"; EXPECTED_SCIPY="1.17.1"; EXPECTED_SKLEARN="1.9.0"

import os, sys, subprocess, importlib, hashlib, json
from pathlib import Path
import numpy as np, scipy, sklearn
assert (np.__version__, scipy.__version__, sklearn.__version__) == (
    EXPECTED_NUMPY, EXPECTED_SCIPY, EXPECTED_SKLEARN
), (np.__version__, scipy.__version__, sklearn.__version__)

os.chdir("/content")
subprocess.run(["rm","-rf","st-guitar-fingering-training"], check=True)
subprocess.run(["git","clone","-q","https://github.com/khfy7wpr5p-maker/st-guitar-fingering-training.git"], check=True)
os.chdir("/content/st-guitar-fingering-training")
subprocess.run(["git","checkout","-q",PINNED_CODE_SHA], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e","."], check=True)
sys.path.insert(0,"/content/st-guitar-fingering-training/src"); importlib.invalidate_caches()
repo_sha=subprocess.check_output(["git","rev-parse","HEAD"], text=True).strip()
assert repo_sha == PINNED_CODE_SHA
print("===== S0-B SETUP PASS =====")
print("Python",sys.version.split()[0],"NumPy",np.__version__,"SciPy",scipy.__version__,"sklearn",sklearn.__version__)
print("Code SHA",repo_sha)
SETUP_READY=True


In [ ]:
# STEP 3 — UPLOAD EXACT CHOICES + PRIOR S0 RESULT
assert SETUP_READY
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 2, "STOP: yalnız choices JSON + önceki S0 result JSON yükle."

by_sha = {hashlib.sha256(data).hexdigest(): (name, data) for name, data in uploaded.items()}
assert EXPECTED_CHOICES_SHA256 in by_sha, f"STOP: choices SHA bulunamadı. Gelen SHA'lar: {list(by_sha)}"
assert EXPECTED_S0_RESULT_SHA256 in by_sha, f"STOP: S0 result SHA bulunamadı. Gelen SHA'lar: {list(by_sha)}"

_, choices_data = by_sha[EXPECTED_CHOICES_SHA256]
_, s0_data = by_sha[EXPECTED_S0_RESULT_SHA256]
choices_path = Path("/content/st-guitar-fingering-training/ST_Guitar_E3_Batch01_choices_400of400.json")
s0_path = Path("/content/st-guitar-fingering-training/ST_Guitar_Stage7G_E3_S0_Diagnostic_result.json")
choices_path.write_bytes(choices_data)
s0_path.write_bytes(s0_data)

reference_s0 = json.loads(s0_data)
assert reference_s0["stage"] == "7G-E3-S0"
assert reference_s0["identity"]["choices_sha256"] == EXPECTED_CHOICES_SHA256
assert reference_s0["identity"]["e3e_teacher_gold_used"] is False
assert reference_s0["identity"]["stage7e_used"] is False
assert reference_s0["checkpoint_retained"] is False
print("===== INPUT SHA PASS =====")
print("Choices:", EXPECTED_CHOICES_SHA256)
print("Prior S0:", EXPECTED_S0_RESULT_SHA256)
INPUTS_READY=True


In [ ]:
# STEP 4 — RECONSTRUCT THE SAME 399 DEVELOPMENT ROWS
assert INPUTS_READY
import platform, urllib.parse, urllib.request
from tempfile import TemporaryDirectory
from st_guitar_fingering_training.target_free_musicxml import parse_target_free_musicxml
from st_guitar_fingering_training.stage7g_e3_e_a3 import reconstruct_frozen_open_low_compact_specialists
from st_guitar_fingering_training.stage7g_e3_r2_learning import build_stage7g_e3_r2_disagreement_pool, rows_from_choices
from st_guitar_fingering_training.stage7g_e3_s0b_error_attribution import STAGE7G_E3_S0B_CONFIG

protocol=json.loads(Path("evidence/stage7g_e3_s0b_error_attribution_protocol.json").read_text())
assert protocol["status"] == "PREREGISTERED_NO_RESULTS"
assert protocol["architecture_decision"]["specialist_architecture_status"] == "TARGET_ARCHITECTURE_CANDIDATE_ONLY"

manifest=json.loads(Path("evidence/stage7g_c_r1_animetab_batch01_manifest.json").read_text())
assert manifest["family_count"] == 40 and manifest["staff_id"] == "2" and manifest["part_id"] == "P1"

sources=[]
with TemporaryDirectory() as tmp:
    root=Path(tmp)
    for i,item in enumerate(manifest["sources"],1):
        url=("https://raw.githubusercontent.com/amamiya-yuuko/AnimeTAB/"
             + ANIMETAB_COMMIT + "/AnimeTAB/Entire%20songs/"
             + urllib.parse.quote(item["filename"], safe=""))
        req=urllib.request.Request(url,headers={"User-Agent":"st-guitar-stage7g-e3-s0b-colab-v1"})
        with urllib.request.urlopen(req,timeout=45) as r:
            raw=r.read()
        assert hashlib.sha256(raw).hexdigest() == item["sha256"]
        p=root/f"{i:03d}.xml"; p.write_bytes(raw)
        sources.append(parse_target_free_musicxml(
            p,
            family_id=item["family_id"],
            tuning=manifest["tuning_midi"],
            pitch_mode=manifest["pitch_mode"],
            part_id=manifest["part_id"],
            staff_id=manifest["staff_id"],
        ))

models,guard=reconstruct_frozen_open_low_compact_specialists()
assert guard["status"] == "PASS_STAGE7B_C2_OPEN_LOW_COMPACT_RECONSTRUCTION"
pool=build_stage7g_e3_r2_disagreement_pool(tuple(sources), specialist_models=models)
rows,preflight=rows_from_choices(pool,json.loads(choices_path.read_text()))
assert preflight["status"] == "R2_PREFLIGHT_PASS_STOP_BEFORE_MANUAL_TRAIN"
assert (preflight["decisive_rows"], preflight["families"], preflight["feature_count"]) == (399,40,40)

identity={
    "code_sha":PINNED_CODE_SHA,
    "choices_sha256":EXPECTED_CHOICES_SHA256,
    "prior_s0_result_sha256":EXPECTED_S0_RESULT_SHA256,
    "animetab_commit":ANIMETAB_COMMIT,
    "python":platform.python_version(),
    "numpy":np.__version__,
    "scipy":scipy.__version__,
    "scikit_learn":sklearn.__version__,
    "e3e_teacher_gold_used":False,
    "stage7e_used":False,
    "checkpoint_retained":False,
    "specialist_architecture_activated":False,
}
print(json.dumps(preflight,indent=2))
print("===== S0-B PREFLIGHT PASS — STOP BEFORE MANUAL AUDIT =====")
PREFLIGHT_READY=True


In [ ]:
# ▶ MANUAL S0-B EVENT AUDIT
assert PREFLIGHT_READY
from st_guitar_fingering_training.stage7g_e3_s0b_error_attribution import stage7g_e3_s0b_event_audit

print("===== S0-B START ===== frozen 5-fold OOF event audit çalışıyor...")
audit=stage7g_e3_s0b_event_audit(rows)

# Scientific continuity gate: reproduce the prior S0 aggregate exactly/tightly.
expected=reference_s0["report"]["oof_final_epoch"]
actual=audit["aggregate_oof"]
for key in ("tp","fp","fn","tn","support","open_low_support","compact_support"):
    assert actual[key] == expected[key], (key, actual[key], expected[key])
for key in ("accuracy","macro_f1","balanced_accuracy","compact_precision","compact_recall",
            "log_loss","average_precision","roc_auc","brier_score","mcc"):
    assert abs(float(actual[key]) - float(expected[key])) <= 1e-12, (key, actual[key], expected[key])

print("===== PRIOR S0 AGGREGATE REPRODUCTION PASS =====")
print("TP/FP/FN/TN:",actual["tp"],actual["fp"],actual["fn"],actual["tn"])
print("Macro-F1:",f"{actual['macro_f1']:.4f}","Balanced Acc:",f"{actual['balanced_accuracy']:.4f}",
      "C-Prec:",f"{actual['compact_precision']:.4f}","C-Rec:",f"{actual['compact_recall']:.4f}")

print()
print("===== ERRORS BY CURRICULUM LEVEL =====")
for level, counts in audit["summary"]["by_curriculum_level"].items():
    print(level, counts)

print()
print("===== ERRORS BY DESCRIPTIVE PRIMARY BUCKET =====")
for bucket, counts in audit["summary"]["by_primary_bucket"].items():
    print(bucket, counts)

errors=[row for row in audit["event_rows"] if row["error_type"] in ("FN","FP")]
assert len(errors) == actual["fn"] + actual["fp"]
print()
print("===== 60 EVENT-LEVEL ERRORS =====")
print("type level family probability bucket | openΔ meanFretΔ spanΔ stringSpanΔ gapsΔ | event_id")
for row in errors:
    d=row["compact_minus_open"]
    print(
        row["error_type"], row["curriculum_level"], row["family_id"],
        f"{row['compact_probability']:.4f}", row["attribution"]["primary_bucket"], "|",
        f"{d['open_note_count']:+.1f}",
        f"{d['mean_positive_fret']:+.2f}",
        f"{d['positive_fret_span']:+.1f}",
        f"{d['string_span']:+.1f}",
        f"{d['internal_string_gaps']:+.1f}", "|",
        row["event_id"]
    )

result={
    "schema":"st-guitar-stage7g-e3-s0b-colab-result-v1",
    "stage":"7G-E3-S0-B",
    "identity":identity,
    "preflight":preflight,
    "audit":audit,
    "prior_s0_aggregate_reproduced":True,
    "checkpoint_retained":False,
    "specialist_architecture_activated":False,
    "e3e_teacher_gold_used":False,
    "stage7e_used":False,
    "production_or_shadow_integration":False,
}
out=Path("ST_Guitar_Stage7G_E3_S0B_Error_Attribution_result.json")
payload=json.dumps(result,indent=2)+"\n"
out.write_text(payload,encoding="utf-8")
print()
print("===== S0-B COMPLETE =====")
print("Result SHA256:",hashlib.sha256(payload.encode()).hexdigest())
files.download(str(out))
